In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
import string
import os
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding
)
from datasets import Dataset
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 1. LOAD DATA & PREPROCESSING (ANGKA DIPERTAHANKAN)
# ============================================================
path = kagglehub.dataset_download("salmanabdu/tokopedia-product-reviews-2025")
df = pd.read_csv(os.path.join(path, 'tokopedia_product_reviews_2025.csv'))
df = df[['review_text', 'sentiment_label']].copy()
df.dropna(inplace=True)

# Filter minimal kata
df['word_count'] = df['review_text'].astype(str).apply(lambda x: len(x.split()))
df = df[df['word_count'] >= 3]

def clean_text_refined(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'&\w+;', '', text)
    # ANGKA TIDAK DIHAPUS agar konteks "Bintang 1/5" tetap ada
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review_text'].apply(clean_text_refined)
df = df[df['clean_review'].str.len() > 0]

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment_label']) # 0: Negatif, 1: Netral, 2: Positif (biasanya)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

# ============================================================
# 2. FOCAL LOSS DENGAN DAMPENED ALPHA
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, num_classes=3):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        # Alpha akan di-move ke GPU secara otomatis oleh Trainer via register_buffer
        if alpha is not None:
            self.register_buffer('alpha', alpha)
        else:
            self.register_buffer('alpha', torch.ones(num_classes))

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        alpha_t = self.alpha[targets]
        focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

class FocalTrainer(Trainer):
    def __init__(self, *args, focal_gamma=2.0, focal_alpha=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma, num_classes=3)
        self.focal_loss.to(self.args.device) # Pastikan di device yang sama

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ============================================================
# 3. PREPARASI MODEL & TRAINING
# ============================================================
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples['text'], padding=False, truncation=True, max_length=128)

train_ds = Dataset.from_pandas(pd.DataFrame({'text': X_train.values, 'label': y_train.values}))
test_ds = Dataset.from_pandas(pd.DataFrame({'text': X_test.values, 'label': y_test.values}))
train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

# Hitung Dampened Alpha (Square Root of Inverse Frequency)
class_counts = Counter(y_train)
counts = [class_counts[i] for i in range(len(class_counts))]
total = sum(counts)
alpha_raw = [np.sqrt(total / c) for c in counts] # Penggunaan Akar (SQRT)
alpha_norm = [a / sum(alpha_raw) * len(counts) for a in alpha_raw]
alpha_tensor = torch.tensor(alpha_norm).float()

print(f"Class Counts: {counts}")
print(f"Dampened Alpha Weights: {alpha_norm}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# Training Args dengan Strategi STEPS untuk Keamanan Colab
training_args = TrainingArguments(
    output_dir='./indobert_focal_v4_1',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,          # Lowered for stability
    weight_decay=0.05,           # Increased for regularization
    eval_strategy="steps",       # Per 200 step agar aman jika disconnect
    save_strategy="steps",
    eval_steps=200,
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_steps=50,
    save_total_limit=2           # Simpan 2 checkpoint terakhir saja agar disk tidak penuh
)

trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=lambda p: {
        'accuracy': accuracy_score(p.label_ids, p.predictions.argmax(-1)),
        'f1_macro': f1_score(p.label_ids, p.predictions.argmax(-1), average='macro')
    },
    focal_gamma=2.0,
    focal_alpha=alpha_tensor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\n🚀 Menjalankan Model 4.1 (Corrected Preprocessing & Dampened Alpha)...")
trainer.train()

# ============================================================
# 4. EVALUASI AKHIR
# ============================================================
predictions = trainer.predict(test_ds)
y_pred = predictions.predictions.argmax(-1)
print("\nClassification Report Akhir:")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Using Colab cache for faster access to the 'tokopedia-product-reviews-2025' dataset.


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Classification Report Akhir:
              precision    recall  f1-score   support

    negative       0.53      0.53      0.53       154
     neutral       0.31      0.45      0.37       151
    positive       0.99      0.99      0.99     11467

    accuracy                           0.97     11772
   macro avg       0.61      0.65      0.63     11772
weighted avg       0.98      0.97      0.98     11772


Confusion Matrix:
[[   81    46    27]
 [   29    68    54]
 [   43   107 11317]]


In [ ]:
tokenizer.save_pretrained('./model_final')

('./model_final/tokenizer_config.json', './model_final/tokenizer.json')